# ЛР4 — дополнительные опыты

GPU T4, Internet On. Один блокнот, три серии по 16 эпизодов на прежних восьми отладочных задачах. Порог 1024 символа, так как 4000 не затрагивает результаты v2. Итоговая отложенная серия не повторяется.

In [ ]:
import os, sys, json, shutil, subprocess, signal, time, urllib.request
from pathlib import Path

ART = Path("/kaggle/working/lab4_optional")
ART.mkdir(parents=True, exist_ok=True)
repo = Path("/tmp/lab4-optional-project")
ENV = dict(os.environ)
ENV.pop("PYTHONPATH", None)
ENV.update(CUDA_VISIBLE_DEVICES="0", HF_HOME="/tmp/lab4-hf", PYTHONNOUSERSITE="1",
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")

def cmd(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], env=ENV, check=True, **kwargs)

gpu = cmd(["nvidia-smi", "--id=0", "--query-gpu=uuid,name,memory.total,driver_version",
           "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert "T4" in gpu, gpu
print("GPU:", gpu)
COMMIT = input("Вставь полный хеш коммита с optional-скриптами: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)
if not repo.exists():
    cmd(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "--single-branch", "--branch", "main",
         "https://github.com/manaer6-alt/intsys-labs.git", repo])
    cmd(["git", "-C", repo, "sparse-checkout", "set", "lab4", "data/agentdojo"])
head = cmd(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if head != COMMIT:
    cmd(["git", "-C", repo, "fetch", "--depth", "1", "origin", COMMIT])
    cmd(["git", "-C", repo, "checkout", "--detach", COMMIT])
assert cmd(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() == COMMIT
for p in ("run_optional.py", "analyze_optional.py"):
    assert (repo / "lab4" / p).exists()
(ART / "commit.txt").write_text(COMMIT + "\n", encoding="utf-8")


In [ ]:
if not shutil.which("uv"):
    cmd([sys.executable, "-m", "pip", "install", "uv"])
uv = shutil.which("uv")
assert uv
PYTHON = "/tmp/lab4-env/bin/python"
AGENT_PYTHON = "/tmp/lab4-agent-env/bin/python"
if not Path(PYTHON).exists():
    cmd([uv, "venv", "--python", "3.13", "--seed", "/tmp/lab4-env"])
cmd([uv, "pip", "install", "--python", PYTHON, "--torch-backend=auto",
     "vllm==0.33.0", "torch==2.14.0+cu132", "transformers==5.20.2",
     "huggingface-hub==2.4.0", "tokenizers==0.23.3", "numpy==2.4.6",
     "openai==3.45.0", "PyYAML==6.0.3", "wrapt==2.5.2"])
cmd([uv, "pip", "check", "--python", PYTHON])
cmd([PYTHON, "-c", "import torch; assert torch.cuda.is_available() and torch.cuda.device_count()==1; print(torch.cuda.get_device_name(0))"])
if not Path(AGENT_PYTHON).exists():
    cmd([uv, "venv", "--python", "3.13", "--seed", "/tmp/lab4-agent-env"])
cmd([uv, "pip", "install", "--python", AGENT_PYTHON, "agentdojo==0.1.35", "wrapt==2.5.2"])
cmd([uv, "pip", "check", "--python", AGENT_PYTHON])
for python, name in ((PYTHON, "requirements_gpu.txt"), (AGENT_PYTHON, "requirements_agent.txt")):
    result = cmd([uv, "pip", "freeze", "--python", python], capture_output=True, text=True)
    (ART / name).write_text(result.stdout, encoding="utf-8")
cmd([AGENT_PYTHON, "-B", "-c", "from run_optional import FROZEN, HERE, sha; assert all(sha(HERE / name)==value for name,value in FROZEN.items()); print('Frozen v2 OK')"], cwd=repo / "lab4")


In [ ]:
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
REV = "cdbee75f17c01a7cc42f958dc650907174af0554"
BASE = "http://127.0.0.1:8000"
server = None

def stop_server():
    global server
    if server is not None and server.poll() is None:
        os.killpg(server.pid, signal.SIGTERM)
        try:
            server.wait(timeout=40)
        except subprocess.TimeoutExpired:
            os.killpg(server.pid, signal.SIGKILL)
            server.wait()
    server = None

def start_server(cache):
    global server
    stop_server()
    flag = "--enable-prefix-caching" if cache else "--no-enable-prefix-caching"
    label = "on" if cache else "off"
    command = ["/tmp/lab4-env/bin/vllm", "serve", MODEL,
               "--revision", REV, "--tokenizer-revision", REV,
               "--dtype", "half", "--max-model-len", "16384",
               "--max-num-seqs", "1", "--max-num-batched-tokens", "2048",
               "--gpu-memory-utilization", "0.9", "--tensor-parallel-size", "1",
               "--attention-backend", "TRITON_ATTN", "--generation-config", "vllm",
               "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
               flag, "--seed", "0", "--host", "127.0.0.1", "--port", "8000"]
    log_path = ART / f"server_{label}.log"
    with log_path.open("w", encoding="utf-8") as log:
        server = subprocess.Popen(command, env=ENV, stdout=log,
                                  stderr=subprocess.STDOUT, start_new_session=True)
    config_path = ART / f"server_{label}.json"
    config_path.write_text(json.dumps({"model": MODEL, "revision": REV,
        "gpu": gpu, "command": command, "commit": COMMIT}, indent=2), encoding="utf-8")
    for i in range(180):
        if server.poll() is not None:
            raise RuntimeError(log_path.read_text(encoding="utf-8", errors="replace")[-6000:])
        try:
            with urllib.request.urlopen(BASE + "/v1/models", timeout=3) as response:
                assert MODEL in [item["id"] for item in json.load(response)["data"]]
                print("Server ready, prefix caching:", cache)
                break
        except OSError:
            time.sleep(5)
    else:
        raise TimeoutError("Server startup timeout")
    # Identical short warmup for both configurations; outside measured episodes.
    body = {"model": MODEL, "messages": [{"role": "user", "content": "Reply with the word ready."}],
            "temperature": 0, "seed": 0, "max_tokens": 16}
    req = urllib.request.Request(BASE + "/v1/chat/completions", data=json.dumps(body).encode(),
                                  headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=180) as response:
        json.load(response)
    return config_path

def run_condition(condition, config_path):
    cmd([AGENT_PYTHON, "-B", repo / "lab4/run_optional.py",
         "--condition", condition, "--out", ART / condition, "--server-config", config_path])


In [ ]:
# 16 эпизодов: базовая серия, кэш выключен.
off = start_server(False)
run_condition("baseline", off)


In [ ]:
# 16 эпизодов: изменяется только длина результата инструмента.
assert server is not None and server.poll() is None
run_condition("truncate_1024", off)
stop_server()


In [ ]:
# 16 эпизодов: полный результат инструмента, кэш включён.
on = start_server(True)
run_condition("prefix_cache", on)
stop_server()


In [ ]:
# Итоговые метрики и архив. Можно выполнить и при остановке неполной серии.
conditions = ("baseline", "truncate_1024", "prefix_cache")
if all((ART / c / "episodes.csv").exists() for c in conditions):
    try:
        cmd([AGENT_PYTHON, "-B", repo / "lab4/analyze_optional.py", "--root", ART])
    except subprocess.CalledProcessError:
        print("Серия неполная или условия не совпали. Сохраняем частичный архив.")
complete = (ART / "optional_summary.json").exists()
name = "/kaggle/working/lab4_optional_results" if complete else "/kaggle/working/lab4_optional_partial"
archive = shutil.make_archive(name, "zip", ART)
print("Download:", archive)
